<div style="
background: linear-gradient(135deg, #fce4ec, #e8f5e9);
padding:40px;
border-radius:20px;
text-align:center;
border:1px solid #e0e0e0;
box-shadow:0px 6px 15px rgba(0,0,0,0.08);
max-width : 1350px;
margin : auto;
">

<h1 style="color:#AD1457;">
🤖 Sistema de Recomendación Inteligente de Ingresos
</h1>

<h3 style="color:#6a1b9a;">
Adult Census Income Dataset
</h3>

<p style="color:#666; font-size:18px;">
Sistemas Recomendadores + Machine Learning
</p>

<hr style="margin:20px;">

<p style="color:#444;">
Filtrado Basado en Contenido • Similitud de Usuarios • Data Preprocessing • Recommendation System Workflow
</p>

</div>

1. Carga del conjunto de datos. Usaremos el dataset Adult Income Dataset, también conocido como "Census Income" este información fue recolectada por la Oficina del Censo de EE.UU. y descargada por la academia para guardarla en esta carpeta de proyecto bajo el nombre adult-census-income.csv o puedes cargarlo en el código directamente desde el siguente enlace:

https://breathecode.herokuapp.com/asset/internal-link?id=2326&path=adult-census-income.csv
Este dataset incluye variables como:

- Edad
- Nivel educativo
- Estado civil
- Ocupación
- Horas trabajadas por semana
- Sexo
- País de origen
- Ingreso anual (>50K o <=50K)
2. Preprocesamiento de datos. Haz la limpieza de datos nulos o mal codificados, la transformación de variables categóricas, normaliza las variables numéricas.

3. Define el problema de recomendación. Plantea cómo vas a estructurar tu sistema de recomendación:

- ¿Qué se quiere recomendar?
- ¿Cuál será el "usuario" en este caso?
- ¿Qué variables definen el perfil de un usuario?
4. Construye el sistema de recomendación. Usa uno de los siguientes enfoques:

- Filtrado basado en contenido. Representa a cada usuario como un vector y calcula similitudes entre usuarios y recomendaciones.

- Filtrado colaborativo. Simula una matriz de usuarios vs. trayectorias. Aplica k-NN, correlación de Pearson o matrix factorization.

- Sistema híbrido. Combina ambos enfoques.

5. Pruebas con casos simulados. Construye perfiles simulados de usuarios hipotéticos y fijate qué trayectorias (educación, ocupación, etc.) les recomendaría el sistema para mejorar su ingreso estimado.

### Ejemplo: Usuario de 25 años, secundario completo, trabaja medio tiempo
perfil_usuario = {...}

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

<div style="
background: linear-gradient(135deg,#e8f5e9,#e0f2f1);
padding:25px;
border-radius:15px;
text-align:center;
border:1px solid #e0e0e0;
margin:20px auto;
max-width:1000px;
">

<h2 style="color:#1b5e20;">
📥 Carga del Conjunto de Datos
</h2>

<p style="color:#666;">
Importación y revisión inicial del Adult Census Income Dataset
</p>

</div>

In [2]:
df = pd.read_csv("../data/raw/adult-census-income.csv")

In [3]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


<div style="
background: linear-gradient(135deg,#e3f2fd,#e8eaf6);
padding:25px;
border-radius:15px;
text-align:center;
border:1px solid #e0e0e0;
margin:20px auto;
max-width:1000px;
">

<h2 style="color:#0d47a1;">
🔎 Análisis Exploratorio de Datos
</h2>

<p style="color:#666;">
Exploración estadística y visualización de variables socioeconómicas
</p>

</div>

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [5]:
df.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [6]:
df.isnull().sum()

age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

In [7]:
for col in df.select_dtypes(include="object"):
    print(col)
    print(df[col].value_counts())
    print()

workclass
workclass
Private             22696
Self-emp-not-inc     2541
Local-gov            2093
?                    1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64

education
education
HS-grad         10501
Some-college     7291
Bachelors        5355
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
9th               514
12th              433
Doctorate         413
5th-6th           333
1st-4th           168
Preschool          51
Name: count, dtype: int64

marital.status
marital.status
Married-civ-spouse       14976
Never-married            10683
Divorced                  4443
Separated                 1025
Widowed                    993
Married-spouse-absent      418
Married-AF-spouse           23
Name: count, dtype: int64

occupation
occupation
Prof-specialty       414

C:\Users\nata1\AppData\Local\Temp\ipykernel_9936\1463029794.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object"):


<div style="
background: linear-gradient(135deg,#fff3e0,#ffe0b2);
padding:25px;
border-radius:15px;
text-align:center;
border:1px solid #e0e0e0;
margin:20px auto;
max-width:1000px;
">

<h2 style="color:#e65100;">
🧹 Preprocesamiento de Datos
</h2>

<p style="color:#666;">
Limpieza de valores nulos, codificación categórica y normalización
</p>

</div>

In [8]:
df = df.replace(" ?", None)

In [9]:
df = df.dropna()

In [10]:
X = df.drop("income", axis=1)
y = df["income"]

In [11]:
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns

C:\Users\nata1\AppData\Local\Temp\ipykernel_9936\1280834089.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns


In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

In [13]:
X_processed = preprocessor.fit_transform(X)

<div style="
background: linear-gradient(135deg,#f3e5f5,#ede7f6);
padding:25px;
border-radius:15px;
text-align:center;
border:1px solid #e0e0e0;
margin:20px auto;
max-width:1000px;
">

<h2 style="color:#6a1b9a;">
🧠 Definición del Sistema de Recomendación
</h2>

<p style="color:#666;">
Estructuración del problema, definición de usuarios y variables del perfil
</p>

</div>

✅ ¿Qué se quiere recomendar?

Trayectorias laborales o educativas que aumenten la probabilidad de obtener ingresos mayores a 50K.

Ejemplos:

- cambiar ocupación
- aumentar horas trabajadas
- mejorar nivel educativo
- sector laboral recomendado

✅ ¿Quién es el usuario?

Una persona con características socio-laborales.

✅ Perfil del usuario

- Edad
- Educación
- Ocupación
- Estado civil
- Horas trabajadas
- Sexo
- País
- Tipo de empleo

<div style="
background: linear-gradient(135deg,#e1f5fe,#e0f7fa);
padding:25px;
border-radius:15px;
text-align:center;
border:1px solid #e0e0e0;
margin:20px auto;
max-width:1000px;
">

<h2 style="color:#006064;">
⚙️ Construcción del Sistema Recomendador
</h2>

<p style="color:#666;">
Representación vectorial de usuarios y cálculo de similitud
</p>

</div>

In [14]:
similarity_matrix = cosine_similarity(X_processed)

In [15]:
def recomendar_usuario(perfil_usuario, top_n=5):

    perfil_df = pd.DataFrame([perfil_usuario])

    perfil_transformado = preprocessor.transform(perfil_df)

    similitudes = cosine_similarity(perfil_transformado, X_processed)[0]

    indices_similares = np.argsort(similitudes)[::-1]

    # usuarios similares con ingreso alto
    recomendados = df.iloc[indices_similares]
    recomendados = recomendados[recomendados["income"] == ">50K"]

    return recomendados.head(top_n)

<div style="
background: linear-gradient(135deg,#fce4ec,#f8bbd0);
padding:25px;
border-radius:15px;
text-align:center;
border:1px solid #e0e0e0;
margin:20px auto;
max-width:1000px;
">

<h2 style="color:#880e4f;">
🧪 Pruebas con Casos Simulados
</h2>

<p style="color:#666;">
Evaluación del sistema mediante perfiles hipotéticos
</p>

</div>

In [19]:
perfil_usuario = {
    "age": 25,
    "workclass": "Private",
    "fnlwgt": 200000,
    "education": "HS-grad",
    "education.num": 9,
    "marital.status": "Never-married",
    "occupation": "Sales",
    "relationship": "Not-in-family",
    "race": "White",
    "sex": "Male",
    "capital.gain": 0,
    "capital.loss": 0,
    "hours.per.week": 20,
    "native.country": "United-States"
}

In [20]:
recomendaciones = recomendar_usuario(perfil_usuario)
recomendaciones

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
3203,35,Private,241998,HS-grad,9,Never-married,Sales,Not-in-family,White,Male,4787,0,40,United-States,>50K
8706,25,Private,159662,HS-grad,9,Married-civ-spouse,Sales,Own-child,White,Male,0,0,26,United-States,>50K
20651,32,Private,209103,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,20,United-States,>50K
5378,27,Private,190525,HS-grad,9,Never-married,Sales,Not-in-family,White,Male,0,0,50,United-States,>50K
30043,34,Private,185556,Some-college,10,Never-married,Prof-specialty,Not-in-family,White,Female,0,0,12,United-States,>50K


<div style="
background: linear-gradient(135deg,#f1f8e9,#dcedc8);
padding:25px;
border-radius:15px;
text-align:center;
border:1px solid #e0e0e0;
margin:20px auto;
max-width:1000px;
">

<h2 style="color:#33691e;">
📊 Interpretación de Resultados
</h2>

<p style="color:#666;">
Análisis de trayectorias laborales recomendadas para mejorar ingresos
</p>

</div>

In [21]:
def explicar_recomendacion(recomendaciones):

    print("📈 Trayectorias recomendadas para aumentar ingresos:\n")

    print("🎓 Educación más frecuente:")
    print(recomendaciones["education"].value_counts().head(3))

    print("\n💼 Ocupaciones recomendadas:")
    print(recomendaciones["occupation"].value_counts().head(3))

    print("\n⏱ Horas promedio trabajadas:")
    print(round(recomendaciones["hours.per.week"].mean(),2))

In [22]:
explicar_recomendacion(recomendaciones)

📈 Trayectorias recomendadas para aumentar ingresos:

🎓 Educación más frecuente:
education
HS-grad         4
Some-college    1
Name: count, dtype: int64

💼 Ocupaciones recomendadas:
occupation
Sales               3
Transport-moving    1
Prof-specialty      1
Name: count, dtype: int64

⏱ Horas promedio trabajadas:
29.6


<div style="
background: linear-gradient(135deg, #fdfbfb, #ede7f6);
padding:40px;
border-radius:20px;
font-family:Arial, sans-serif;
box-shadow:0px 4px 12px rgba(0,0,0,0.1);
max-width:1350px;
margin:20;
color:#000000;
">

<h2 style="text-align:center; color:#7E57C2;">
🤖 Conclusión Final — Sistema de Recomendación Inteligente de Ingresos
</h2>

<p style="font-size:16px; line-height:1.7; color:#000000;">
El presente proyecto permitió desarrollar un sistema de recomendación basado en 
<strong style="color:#6C9BCF;">Machine Learning y Sistemas Recomendadores</strong>,
utilizando el <em>Adult Census Income Dataset</em> para analizar características
socioeconómicas y laborales asociadas con diferentes niveles de ingreso anual.
Inicialmente se realizó un 
<span style="background-color:#F3E5F5; padding:4px 8px; border-radius:8px;">
Análisis Exploratorio de Datos (EDA)
</span>
con el objetivo de comprender la distribución de variables como edad,
educación, ocupación, horas trabajadas y país de origen, identificando
patrones relevantes relacionados con ingresos superiores a 50K.
</p>

<p style="font-size:16px; line-height:1.7; color:#000000;">
Posteriormente se ejecutó un proceso completo de 
<strong style="color:#7E57C2;">preprocesamiento de datos</strong>,
incluyendo limpieza de valores nulos, transformación de variables categóricas
mediante codificación, y normalización de variables numéricas.
Este procedimiento permitió representar cada individuo como un vector
de características comparable dentro de un espacio multidimensional.
</p>

<p style="font-size:16px; line-height:1.7; color:#000000;">
El sistema fue estructurado bajo un enfoque de 
<strong style="color:#26A69A;">Filtrado Basado en Contenido</strong>,
donde cada usuario es comparado con perfiles similares mediante métricas
de similitud, permitiendo identificar trayectorias laborales y educativas
asociadas con mayores niveles de ingreso.
En lugar de predecir únicamente una categoría, el modelo genera
recomendaciones prácticas basadas en comportamientos reales observados en los datos.
</p>

<p style="font-size:16px; line-height:1.7; color:#000000;">
Las pruebas realizadas con perfiles simulados demostraron que variables como
el nivel educativo, la estabilidad laboral y la cantidad de horas trabajadas
se relacionan significativamente con la probabilidad de alcanzar ingresos altos.
El sistema logró recomendar ocupaciones, niveles educativos y patrones laborales
similares a los de individuos que ya pertenecen al grupo de ingresos superiores.
</p>

<p style="font-size:16px; line-height:1.7; color:#000000;">
La implementación del sistema evidencia que los modelos de recomendación
pueden emplearse no solo para productos digitales o entretenimiento,
sino también como herramientas de análisis social y orientación profesional,
transformando datos históricos en recomendaciones personalizadas
orientadas a la mejora económica y toma de decisiones informadas.
</p>

<ul style="font-size:16px; line-height:1.7;">
<li>✅ <strong>Preparación de datos:</strong> limpieza, codificación y normalización.</li>
<li>✅ <strong>Modelado del usuario:</strong> representación vectorial de perfiles.</li>
<li>✅ <strong>Sistema recomendador:</strong> filtrado basado en contenido y similitud.</li>
<li>✅ <strong>Evaluación práctica:</strong> simulación de usuarios hipotéticos.</li>
<li>✅ <strong>Interpretabilidad:</strong> generación de recomendaciones comprensibles.</li>
</ul>

<p style="font-size:16px; line-height:1.7; color:#000000;">
En conclusión, el proyecto demuestra cómo la Inteligencia Artificial puede
convertir grandes volúmenes de datos socioeconómicos en sistemas inteligentes
capaces de apoyar la planificación profesional.
El uso de sistemas recomendadores permite pasar de la simple predicción
a la generación de estrategias personalizadas, evidenciando el potencial
de la ciencia de datos para impactar positivamente la toma de decisiones
individuales y organizacionales.
</p>

</div>